# 03 — Interpolating poll-level election results to census tracts

The three per-poll result tables built in notebook 01 (`municipal_2023_mayor_polls.csv`,
`provincial_2025_polls.csv`, `federal_2025_polls.csv`) live on polling-subdivision/riding
geography — not census geography. To join election outcomes to the Block 1–3 census profile
built in notebook 02, every poll's votes need to be reallocated onto the 585 census tracts (CTs)
that make up the DA-weighted interpolation universe (`in_interpolation_universe` in
`ct_census_profile.csv`).

This notebook ports the population-weighted spatial-allocation algorithm from the archived
`interpolation/scripts/spatial.py` + `workflow.py` (GDAL/OGR-based) onto the same inputs, but
implemented with `geopandas`/`shapely` (GEOS-based, same underlying geometry engine as GDAL's
OGR bindings) instead of raw `osgeo.ogr`. The core idea, unchanged from the archived pipeline:

1. Reproject everything to a metres-based CRS (`EPSG:3347`, Statistics Canada Lambert) so area
   arithmetic is meaningful.
2. Repair any invalid polygons (`shapely.make_valid`, the same GEOS `MakeValid` the archived
   pipeline calls through OGR).
3. For each poll's polygon (or, for polls with **no** polygon, its riding/ward polygon), intersect
   against the 585 target CTs, and *within* each CT-piece, intersect against the dissemination
   areas (DAs) belonging to that CT, weighting each DA-piece by
   `citizen_canadian_18over * (piece_area / da_area)`.
4. Normalize the resulting CT-weights (population-weighted, or area-weighted as a fallback) so
   they sum to 1 across every CT the poll touches, then multiply votes/electors/valid-votes/party
   votes by that weight and add into the target CT.

This notebook also computes **Block 4** (electoral competitiveness) directly from the freshly
interpolated candidate/party vote estimates: top-two margin, effective number of
candidates/parties (`1/HHI`), and vote fragmentation (`1-HHI`), each using a 5%-vote-share
screening threshold — porting `competition()` from
`analysis/archive/toronto_election_turnout/archive/variables/scripts/build_blocks_1_5_master.py`.

**Inputs used (all read-only):**
- `data/toronto_election_turnout/elections/{municipal_2023_mayor,provincial_2025,federal_2025}_polls.{csv,geojson}`
  — notebook 01's output. The CSV carries the vote/elector/party columns; the GeoJSON carries the
  matching polygon (or `null`) per `poll_id`.
- `data/toronto_election_turnout/census/ct_census_profile.geojson` — notebook 02's
  output, filtered to `in_interpolation_universe == True` (585 CTs). It already carries
  `citizen_canadian_18over` and a `canadian_citizens_18plus_status` suppression flag per CT, so it
  is used directly rather than falling back to the archived CT geojson (both checked and found
  equivalent for this purpose — see the sanity check in Section 1).
- `data/toronto_election_turnout/archive/census/processed/da/statcan_2021_toronto_da.geojson` — DA
  geometry, **already carrying `citizen_canadian_18over` and `value_status`** (i.e. the DA-level
  citizenship intermediate is already joined into this file upstream). There is no notebook
  01/02 equivalent for DA geometry, so this archived-but-still-current processed file is read
  directly, exactly as the plan anticipates for inputs with "no v2 equivalent."
- Riding/ward polygons for the no-geometry fallback, per the archived `interpolation/scripts/config.py`:
  `src/data/wards.geo.json` (municipal wards, field `num`), `data/toronto_election_turnout/archive/elections/raw/ont_2025_ridings.geojson`
  (field `RIDINGNO`), `data/toronto_election_turnout/archive/elections/raw/fed_2025_ridings.geojson` (field `FED_NUM`).
- **One deliberate deviation:** notebook 01's per-poll table collapses the 2023 mayoral race's
  102 candidates into a single `party_non_partisan_votes` column (correctly — it *is* a
  non-partisan race), which is exactly right for the turnout table but leaves no way to compute
  *candidate-level* competitiveness for the mayoral race. That per-poll/per-candidate breakdown
  was never carried into notebook 01's rebuilt output, so this notebook reads the archived
  `elections/processed/municipal_2023_mayor/candidate_details/toronto_municipal_2023_mayor_poll_candidate_votes.csv`
  (+ `..._candidates.csv`) directly, read-only, purely to get the candidate-level Block 4 margin
  right. Provincial/federal Block 4 needs no such workaround — their party columns already live in
  notebook 01's output.

**Ground truth for verification (read-only, not modified):**
`data/toronto_election_turnout/archive/interpolation/processed/*_ct_estimated_results.csv` /
`*_ct_candidate_estimated_votes.csv`, and the `block4_*`/`outcome_*` columns of
`data/toronto_election_turnout/archive/variables/processed/toronto_ct_blocks_1_5_modelling_master.csv`.

In [1]:
import warnings
from collections import defaultdict
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely import STRtree, area, intersection, make_valid
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60)

In [2]:
REPO_ROOT = Path.cwd().resolve().parents[1]
DATA_ROOT = REPO_ROOT / "data" / "toronto_election_turnout"
ARCHIVE_ROOT = DATA_ROOT / "archive"

# Notebook 01 / 02 outputs (this pipeline's own inputs)
ELECTIONS_V2_DIR = DATA_ROOT / "elections"
CT_PROFILE_GEOJSON = DATA_ROOT / "census" / "ct_census_profile.geojson"

# Archived, still-current inputs with no notebook-01/02 equivalent (read-only)
DA_GEOJSON = ARCHIVE_ROOT / "census" / "processed" / "da" / "statcan_2021_toronto_da.geojson"
CT_GEOJSON_ARCHIVE = ARCHIVE_ROOT / "census" / "processed" / "ct" / "statcan_2021_toronto_ct.geojson"
MUNICIPAL_CANDIDATE_VOTES_CSV = (
    ARCHIVE_ROOT / "elections" / "processed" / "municipal_2023_mayor" / "candidate_details"
    / "toronto_municipal_2023_mayor_poll_candidate_votes.csv"
)
MUNICIPAL_CANDIDATE_LOOKUP_CSV = (
    ARCHIVE_ROOT / "elections" / "processed" / "municipal_2023_mayor" / "candidate_details"
    / "toronto_municipal_2023_mayor_candidates.csv"
)

# Ground truth for the final verification cell (read-only, never written to)
INTERP_GT_DIR = ARCHIVE_ROOT / "interpolation" / "processed"
VARIABLES_MASTER_GT_CSV = ARCHIVE_ROOT / "variables" / "processed" / "toronto_ct_blocks_1_5_modelling_master.csv"

# Output tree for this notebook
OUT_DIR = DATA_ROOT / "interpolation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ANCILLARY_WEIGHT_FIELD = "citizen_canadian_18over"
WORKING_EPSG = 3347  # Statistics Canada Lambert Conformal Conic — metres, good for area math over Canada
TOLERANCE = 1e-6      # conservation tolerance for allocation-weight sums (matches archived TOLERANCE)
REPORT_ATOL = 1e-3    # looser "is this basically zero" tolerance used only for reporting/printing vote-count diffs
COMPETITIVENESS_THRESHOLD = 0.05  # vote-share floor for "who counts" in effective-candidates / fragmentation

# election_id -> (poll csv/geojson stem, district geojson, district id field, zero-padded width)
ELECTIONS = {
    "municipal_2023_mayor": {
        "short_name": "municipal",
        "poll_csv": ELECTIONS_V2_DIR / "municipal_2023_mayor_polls.csv",
        "poll_geojson": ELECTIONS_V2_DIR / "municipal_2023_mayor_polls.geojson",
        "district_geojson": REPO_ROOT / "src" / "data" / "wards.geo.json",
        "district_id_field": "num",
        "district_id_width": 2,
    },
    "provincial_2025": {
        "short_name": "provincial",
        "poll_csv": ELECTIONS_V2_DIR / "provincial_2025_polls.csv",
        "poll_geojson": ELECTIONS_V2_DIR / "provincial_2025_polls.geojson",
        "district_geojson": ARCHIVE_ROOT / "elections" / "raw" / "ont_2025_ridings.geojson",
        "district_id_field": "RIDINGNO",
        "district_id_width": 3,
    },
    "federal_2025": {
        "short_name": "federal",
        "poll_csv": ELECTIONS_V2_DIR / "federal_2025_polls.csv",
        "poll_geojson": ELECTIONS_V2_DIR / "federal_2025_polls.geojson",
        "district_geojson": ARCHIVE_ROOT / "elections" / "raw" / "fed_2025_ridings.geojson",
        "district_id_field": "FED_NUM",
        "district_id_width": 5,
    },
}

## 1. The CT/DA geometry universe

The archived pipeline's `run_interpolation.py` hard-asserts that exactly **585** Toronto CT
polygons load before doing anything else — that assertion is kept here verbatim. Notebook 02's
`ct_census_profile.geojson` is used as the CT source (per the rebuild plan's preference), filtered
to `in_interpolation_universe == True`; a quick equivalence check against the archived
`statcan_2021_toronto_ct.geojson`'s `contains_toronto_da` flag confirms the two describe the same
585-CT universe before relying on the newer file.

Every polygon (CTs and DAs) gets reprojected to `EPSG:3347` and repaired with `make_valid` up
front — invalid input polygons are common in StatCan boundary files (self-intersections from
generalization), and an invalid polygon silently produces wrong or zero intersection areas.

In [3]:
ct_gdf = gpd.read_file(CT_PROFILE_GEOJSON)
ct_gdf = ct_gdf.loc[ct_gdf["in_interpolation_universe"] == True].copy()  # noqa: E712
assert len(ct_gdf) == 585, f"Expected 585 Toronto CT polygons in the interpolation universe, found {len(ct_gdf)}"

# Cross-check against the archived CT file's own universe flag, since both are candidate sources.
ct_gdf_archive = gpd.read_file(CT_GEOJSON_ARCHIVE)
archive_universe = set(ct_gdf_archive.loc[ct_gdf_archive["contains_toronto_da"] == True, "geo_id"].astype(float).round(2))
new_universe = set(ct_gdf["ct_id"].astype(float).round(2))
assert archive_universe == new_universe, "notebook 02's CT universe disagrees with the archived contains_toronto_da universe!"
print("CT universe check: notebook 02's 585-CT `in_interpolation_universe` matches the archived `contains_toronto_da` universe exactly.")

ct_gdf["ct_id_key"] = ct_gdf["ct_id"].map(lambda x: f"{x:.2f}")
ct_gdf = ct_gdf.to_crs(WORKING_EPSG).reset_index(drop=True)
ct_gdf["geometry"] = ct_gdf.geometry.apply(make_valid)

da_gdf = gpd.read_file(DA_GEOJSON)
da_gdf["ct_id_key"] = da_gdf["ct_id"].astype(float).map(lambda x: f"{x:.2f}")
da_gdf = da_gdf.to_crs(WORKING_EPSG).reset_index(drop=True)
da_gdf["geometry"] = da_gdf.geometry.apply(make_valid)
da_gdf["area_m2"] = da_gdf.geometry.area
da_gdf["citizen_weight"] = da_gdf[ANCILLARY_WEIGHT_FIELD].fillna(0.0)
da_gdf["citizen_suppressed"] = da_gdf[ANCILLARY_WEIGHT_FIELD].isna()

# Every DA's assigned ct_id should fall inside the 585-CT universe (StatCan DAs nest inside CTs).
unmatched_das = (~da_gdf["ct_id_key"].isin(set(ct_gdf["ct_id_key"]))).sum()
print(f"{len(ct_gdf)} target CTs, {len(da_gdf)} DAs loaded; {unmatched_das} DAs fall outside the CT universe (should be 0).")
print(f"{da_gdf['citizen_suppressed'].sum()} DAs have a confidentiality-suppressed {ANCILLARY_WEIGHT_FIELD} (treated as zero weight, not dropped).")

CT universe check: notebook 02's 585-CT `in_interpolation_universe` matches the archived `contains_toronto_da` universe exactly.


585 target CTs, 3743 DAs loaded; 0 DAs fall outside the CT universe (should be 0).
16 DAs have a confidentiality-suppressed citizen_canadian_18over (treated as zero weight, not dropped).


## 2. The population-weighted spatial crosswalk

**Why proration through DAs, rather than just splitting a poll's votes across CTs by raw
geometric area?** A poll's polygon can straddle a park, a ravine, an arterial road corridor, and a
dense residential block all inside the same CT boundary crossing. Splitting purely by area assumes
population is smeared uniformly across the poll's footprint — which is almost never true. DAs are
StatCan's smallest published geography (typically 400–700 people), so weighting each fragment of a
poll's polygon by the DA population (here, `citizen_canadian_18over`, the closest available proxy
for the eligible-voter population) that actually falls in that fragment captures real intra-poll,
intra-CT population concentration instead of erasing it.

**The fallback:** if a poll/riding's *entire* footprint has zero total population weight — every
DA it overlaps is either genuinely empty or has a suppressed weight treated as zero — there's no
population signal left to divide by. In that case (and only for polls with real geometry;
no-geometry riding/ward fallbacks do **not** get this second fallback, matching the archived
`allow_area_fallback=False` for that path) the weights fall back to pure area-proration across the
overlapping CTs. That's a "better than dropping the votes" choice, not a claim that population is
actually uniform there.

Implementation note: this is the same three-step nested overlay as `spatial.py`'s
`build_population_weights` (source → CT-piece → DA-piece-within-that-CT), just expressed with
`shapely`'s vectorized `STRtree` for candidate lookup instead of OGR's custom envelope grid — same
GEOS engine underneath, so results should agree with the archived GDAL/OGR output to within
floating-point noise (confirmed in Section 6).

In [4]:
ct_tree = STRtree(ct_gdf.geometry.values)
da_tree = STRtree(da_gdf.geometry.values)

ct_ids = ct_gdf["ct_id_key"].values
ct_geoms = ct_gdf.geometry.values
da_ct_ids = da_gdf["ct_id_key"].values
da_geoms = da_gdf.geometry.values
da_areas = da_gdf["area_m2"].values
da_weights = da_gdf["citizen_weight"].values


def population_weights_for_geometry(geom, allow_area_fallback: bool):
    '''Population-weighted (with optional area-fallback) CT allocation weights for one source polygon.

    Returns a list of {"ct_id", "allocation_weight", ...} dicts summing to 1 across the CTs the
    geometry overlaps (or an empty list if it overlaps none), plus a small diagnostics dict.
    '''
    overlaps = []
    total_population_weight = 0.0
    total_ct_area = 0.0
    for ct_idx in ct_tree.query(geom, predicate="intersects"):
        ct_piece = intersection(geom, ct_geoms[ct_idx])
        ct_piece_area = area(ct_piece)
        if ct_piece_area <= 0:
            continue
        total_ct_area += ct_piece_area
        target_ct_id = ct_ids[ct_idx]

        population_weight = 0.0
        for da_idx in da_tree.query(ct_piece, predicate="intersects"):
            if da_ct_ids[da_idx] != target_ct_id:
                continue  # DA belongs to a different CT than the one we're currently accumulating into
            da_piece_area = area(intersection(ct_piece, da_geoms[da_idx]))
            if da_piece_area <= 0:
                continue
            da_area_total = da_areas[da_idx]
            if da_area_total > 0:
                population_weight += da_weights[da_idx] * da_piece_area / da_area_total

        total_population_weight += population_weight
        overlaps.append({"ct_id": target_ct_id, "population_weight": population_weight, "ct_piece_area": ct_piece_area})

    zero_population_weight = total_population_weight <= 0
    use_area_fallback = zero_population_weight and allow_area_fallback and bool(overlaps)
    denominator = total_ct_area if use_area_fallback else total_population_weight
    for row in overlaps:
        numerator = row["ct_piece_area"] if use_area_fallback else row["population_weight"]
        row["allocation_weight"] = numerator / denominator if denominator else 0.0

    diagnostics = {"zero_population_weight": zero_population_weight, "fallback_area_weight_used": use_area_fallback}
    return overlaps, diagnostics

## 3. Loading each election's poll-level sources

Each poll table gets its polygon (or `None`) merged in from the matching GeoJSON on `poll_id`.
District IDs are zero-padded to match the width used by their riding/ward polygon source (municipal
wards: 2 digits, Ontario ridings: 3, federal ridings: 5 — federal's already includes the `35`
province prefix). Rows with `number_of_votes` missing (advance/mail-in totals folded into another
division, per notebook 01's `vote_in_other_division` handling) are excluded from allocation
entirely, same as the archived pipeline.

In [5]:
def normalize_district_id(value, width: int) -> str:
    text = "" if pd.isna(value) else str(value).strip()
    if not text:
        return ""
    try:
        text = str(int(float(text)))
    except ValueError:
        pass
    return text.zfill(width)


def load_election_polls(election_id: str):
    cfg = ELECTIONS[election_id]
    polls = pd.read_csv(cfg["poll_csv"], dtype=str)
    geo = gpd.read_file(cfg["poll_geojson"]).to_crs(WORKING_EPSG)
    geo["geometry"] = geo.geometry.apply(lambda g: make_valid(g) if g is not None else None)
    polls = polls.merge(geo[["poll_id", "geometry"]], on="poll_id", how="left")

    party_fields = [c for c in polls.columns if c.startswith("party_") and c.endswith("_votes")]
    numeric_cols = ["number_of_votes", "number_of_electors", "poll_total_candidate_votes", *party_fields]
    for col in numeric_cols:
        polls[col] = pd.to_numeric(polls[col], errors="coerce")
    polls["district_id"] = polls["electoral_district_number"].map(lambda v: normalize_district_id(v, cfg["district_id_width"]))
    return polls, party_fields


def load_district_geometries(election_id: str, required_ids):
    cfg = ELECTIONS[election_id]
    dist_gdf = gpd.read_file(cfg["district_geojson"]).to_crs(WORKING_EPSG)
    dist_gdf["district_id"] = dist_gdf[cfg["district_id_field"]].map(lambda v: normalize_district_id(v, cfg["district_id_width"]))
    dist_gdf = dist_gdf.loc[dist_gdf["district_id"].isin(required_ids)].copy()
    dist_gdf["geometry"] = dist_gdf.geometry.apply(make_valid)
    return dict(zip(dist_gdf["district_id"], dist_gdf.geometry))

## 4. Allocation: poll → CT, and (for polls with no polygon) riding/ward → CT

Two source types feed each CT's totals, exactly mirroring `workflow.py::run_election`:

- **Mapped polls** (have a polygon): each poll is allocated individually, with
  `allow_area_fallback=True` — a poll with a real (if imperfect) polygon should never be dropped
  just because it happens to sit entirely on suppressed-weight DAs.
- **No-geometry polls** (municipal's institutional/mail-in/advance reporting buckets 96-99;
  provincial/federal advance-poll rows folded into a riding-wide total; see notebook 01's markdown
  on this): grouped by district, the *riding/ward polygon's* population weights are computed
  **once per district** (not once per poll — every no-geometry poll in the same district reuses
  the same normalized weights) with `allow_area_fallback=False`. If a district's population weight
  is entirely zero, its no-geometry votes are excluded rather than smeared arbitrarily by area —
  a coarser fallback than a poll-level one would be too crude to justify at riding scale.

The crosswalk (`poll_id`/`district_id` → `ct_id` → `allocation_weight`) is kept in memory so
Section 6's candidate-level allocation (for the municipal race) can reuse it directly.

In [6]:
def allocate_election(election_id: str):
    cfg = ELECTIONS[election_id]
    polls, party_fields = load_election_polls(election_id)

    ct_accum = {
        cid: {"total_votes": 0.0, "electors": 0.0, "valid_votes": 0.0, **{f: 0.0 for f in party_fields}}
        for cid in ct_ids
    }
    source_totals = defaultdict(float)
    allocated_totals = defaultdict(float)
    crosswalk_rows = []

    def add_allocation(row, ct_id, weight):
        acc = ct_accum[ct_id]
        votes = row["number_of_votes"]
        acc["total_votes"] += votes * weight
        allocated_totals["total_votes"] += votes * weight
        if not pd.isna(row["number_of_electors"]):
            acc["electors"] += row["number_of_electors"] * weight
            allocated_totals["electors"] += row["number_of_electors"] * weight
        if not pd.isna(row["poll_total_candidate_votes"]):
            acc["valid_votes"] += row["poll_total_candidate_votes"] * weight
            allocated_totals["valid_votes"] += row["poll_total_candidate_votes"] * weight
        for field in party_fields:
            if not pd.isna(row[field]):
                acc[field] += row[field] * weight
                allocated_totals[field] += row[field] * weight

    included = polls.loc[polls["number_of_votes"].notna()].copy()
    excluded_missing_votes = int(polls["number_of_votes"].isna().sum())
    for _, row in included.iterrows():
        source_totals["total_votes"] += row["number_of_votes"]
        if not pd.isna(row["number_of_electors"]):
            source_totals["electors"] += row["number_of_electors"]
        if not pd.isna(row["poll_total_candidate_votes"]):
            source_totals["valid_votes"] += row["poll_total_candidate_votes"]
        for field in party_fields:
            if not pd.isna(row[field]):
                source_totals[field] += row[field]

    mapped = included.loc[included["geometry"].notna()]
    no_geometry = included.loc[included["geometry"].isna()]

    for _, row in tqdm(mapped.iterrows(), total=len(mapped), desc=f"{election_id}: mapped polls"):
        overlaps, diagnostics = population_weights_for_geometry(row["geometry"], allow_area_fallback=True)
        for overlap in overlaps:
            add_allocation(row, overlap["ct_id"], overlap["allocation_weight"])
            crosswalk_rows.append((row["poll_id"], overlap["ct_id"], overlap["allocation_weight"]))

    no_geometry_district_ids = sorted(set(no_geometry["district_id"]) - {""})
    district_geoms = load_district_geometries(election_id, set(no_geometry_district_ids))
    district_weights = {}
    excluded_zero_weight_districts = []
    for district_id in tqdm(no_geometry_district_ids, desc=f"{election_id}: no-geometry districts"):
        geom = district_geoms.get(district_id)
        if geom is None:
            continue
        overlaps, diagnostics = population_weights_for_geometry(geom, allow_area_fallback=False)
        district_weights[district_id] = (overlaps, diagnostics)
        if diagnostics["zero_population_weight"] or not overlaps:
            excluded_zero_weight_districts.append(district_id)

    excluded_no_geometry_rows = 0
    for _, row in no_geometry.iterrows():
        weights = district_weights.get(row["district_id"])
        if not row["district_id"] or weights is None:
            excluded_no_geometry_rows += 1
            continue
        overlaps, diagnostics = weights
        if diagnostics["zero_population_weight"] or not overlaps:
            excluded_no_geometry_rows += 1
            continue
        for overlap in overlaps:
            add_allocation(row, overlap["ct_id"], overlap["allocation_weight"])
            crosswalk_rows.append((row["poll_id"], overlap["ct_id"], overlap["allocation_weight"]))

    crosswalk_df = pd.DataFrame(crosswalk_rows, columns=["poll_id", "ct_id", "weight"])
    diagnostics_summary = {
        "input_poll_rows": len(polls),
        "excluded_missing_votes_rows": excluded_missing_votes,
        "mapped_rows": len(mapped),
        "no_geometry_vote_bearing_rows": len(no_geometry),
        "excluded_no_geometry_rows": excluded_no_geometry_rows,
        "zero_weight_no_geometry_districts": excluded_zero_weight_districts,
    }
    return ct_accum, party_fields, crosswalk_df, source_totals, allocated_totals, diagnostics_summary


allocation_results = {}
for election_id in ELECTIONS:
    print(f"--- {election_id} ---")
    result = allocate_election(election_id)
    allocation_results[election_id] = result
    print({k: v for k, v in result[5].items() if k != "zero_weight_no_geometry_districts"})

--- municipal_2023_mayor ---


municipal_2023_mayor: mapped polls:   0%|          | 0/1351 [00:00<?, ?it/s]

municipal_2023_mayor: no-geometry districts:   0%|          | 0/25 [00:00<?, ?it/s]

{'input_poll_rows': 1545, 'excluded_missing_votes_rows': 94, 'mapped_rows': 1351, 'no_geometry_vote_bearing_rows': 100, 'excluded_no_geometry_rows': 0}
--- provincial_2025 ---


provincial_2025: mapped polls:   0%|          | 0/1388 [00:00<?, ?it/s]

provincial_2025: no-geometry districts:   0%|          | 0/25 [00:00<?, ?it/s]

{'input_poll_rows': 1532, 'excluded_missing_votes_rows': 0, 'mapped_rows': 1388, 'no_geometry_vote_bearing_rows': 144, 'excluded_no_geometry_rows': 0}
--- federal_2025 ---


federal_2025: mapped polls:   0%|          | 0/4273 [00:00<?, ?it/s]

federal_2025: no-geometry districts:   0%|          | 0/24 [00:00<?, ?it/s]

{'input_poll_rows': 5069, 'excluded_missing_votes_rows': 382, 'mapped_rows': 4273, 'no_geometry_vote_bearing_rows': 414, 'excluded_no_geometry_rows': 0}


## 5. Validation: vote/elector/party totals are conserved

Because every source's overlap weights are normalized to sum to 1 across the CTs it touches
(Section 2), the sum of allocated votes across all 585 CTs must equal the sum of source votes
across all included polls/districts, for every measure (total votes, electors, valid candidate
votes, and every party column) — this is the same global conservation check `workflow.py` runs
(`TOLERANCE = 1e-6`), asserted here for real rather than just described.

In [7]:
for election_id, (ct_accum, party_fields, crosswalk_df, source_totals, allocated_totals, diag) in allocation_results.items():
    print(f"--- {election_id} conservation check ---")
    for measure in ["total_votes", "electors", "valid_votes", *party_fields]:
        source_value = source_totals[measure]
        allocated_value = allocated_totals[measure]
        difference = abs(source_value - allocated_value)
        # Vote/elector totals run into the hundreds of thousands; TOLERANCE is an absolute bound on
        # the *weight* arithmetic, so scale it by the magnitude of the measure being conserved.
        scaled_tolerance = max(TOLERANCE, TOLERANCE * abs(source_value))
        assert difference <= scaled_tolerance, (
            f"{election_id}: {measure} not conserved -- source={source_value}, allocated={allocated_value}, diff={difference}"
        )
    print(f"  total_votes: source={source_totals['total_votes']:.6f}  allocated={allocated_totals['total_votes']:.6f}  "
          f"diff={abs(source_totals['total_votes'] - allocated_totals['total_votes']):.2e}")
    print(f"  all {2 + len(party_fields)} measures (electors, valid_votes, {len(party_fields)} party columns) conserved within tolerance. PASS")

--- municipal_2023_mayor conservation check ---
  total_votes: source=724638.000000  allocated=724638.000000  diff=1.63e-09
  all 3 measures (electors, valid_votes, 1 party columns) conserved within tolerance. PASS
--- provincial_2025 conservation check ---
  total_votes: source=890829.000000  allocated=890829.000000  diff=1.16e-10
  all 17 measures (electors, valid_votes, 15 party columns) conserved within tolerance. PASS
--- federal_2025 conservation check ---
  total_votes: source=1318564.000000  allocated=1318564.000000  diff=2.56e-09
  all 15 measures (electors, valid_votes, 13 party columns) conserved within tolerance. PASS


## 6. Turnout and participation per CT

`estimated_participation_citizen_18plus` — allocated total votes divided by the CT's
`citizen_canadian_18over` — is the primary turnout measure downstream (the archived pipeline's
diagnostic comparison in `workflow.py` found this Census-based denominator tracks each election's
*published* turnout rate more closely than the raw registered-elector count, which the interpolated
poll-level `number_of_electors` sums are known to under-report for advance/institutional polls).
`outcome_mean_participation_citizen_18plus` averages the three available election participation
rates per CT and is the eventual PLS model's regression target in notebook 05.

In [8]:
ct_citizen_weight = dict(zip(ct_gdf["ct_id_key"], ct_gdf[ANCILLARY_WEIGHT_FIELD]))

outcome_rows = {cid: {"ct_id": cid} for cid in ct_ids}
for election_id, (ct_accum, party_fields, *_rest) in allocation_results.items():
    short_name = ELECTIONS[election_id]["short_name"]
    for cid, acc in ct_accum.items():
        citizen_weight = ct_citizen_weight.get(cid)
        participation = acc["total_votes"] / citizen_weight if citizen_weight and citizen_weight > 0 else None
        turnout_electors = acc["total_votes"] / acc["electors"] if acc["electors"] > 0 else None
        row = outcome_rows[cid]
        row[f"outcome_{short_name}_participation_citizen_18plus"] = participation
        row[f"outcome_{short_name}_turnout_electors"] = turnout_electors
        row[f"{short_name}_estimated_total_votes"] = acc["total_votes"]
        row[f"{short_name}_estimated_electors"] = acc["electors"]

participation_cols = [f"outcome_{cfg['short_name']}_participation_citizen_18plus" for cfg in ELECTIONS.values()]
for cid, row in outcome_rows.items():
    present = [row[col] for col in participation_cols if row.get(col) is not None]
    row["outcome_mean_participation_citizen_18plus"] = float(np.mean(present)) if present else None

outcome_df = pd.DataFrame(outcome_rows.values())
outcome_df.describe()[participation_cols + ["outcome_mean_participation_citizen_18plus"]]

,outcome_municipal_participation_citizen_18plus,outcome_provincial_participation_citizen_18plus,outcome_federal_participation_citizen_18plus,outcome_mean_participation_citizen_18plus
count,583.000000,583.000000,583.000000,583.000000
mean,0.389193,0.474944,0.703542,0.522560
std,0.103915,0.092496,0.099594,0.087149
min,0.101570,0.221357,0.476543,0.304110
25%,0.313532,0.409473,0.633377,0.458906
50%,0.374222,0.472874,0.695730,0.521123
75%,0.462351,0.536543,0.768245,0.589951
max,0.826836,0.973359,1.211840,0.979253


## 7. Block 4: electoral competitiveness

Three related but distinct signals, computed per CT per election from the freshly interpolated
vote shares:

- **Margin** (top-two vote-share gap) — the most literal read of "how close was this race here,"
  using *every* candidate/party with nonzero votes (no threshold).
- **Effective number of candidates/parties** (`1/HHI`, the inverse Herfindahl-Hirschman Index) —
  answers "how many equally-sized competitors would produce this same level of vote fragmentation?"
  A 2-effective-candidate CT is a clean two-way race; a 4-effective-candidate CT is genuinely
  fragmented even if no single challenger looks dominant.
- **Fragmentation** (`1 - HHI`) — the complement of the same index, read as "how much of the vote
  is *not* concentrated in one place," a smoother 0-1 diversity score than a raw candidate count.

Both HHI-based measures use a **5%-vote-share screening threshold**: without it, the 2023 mayoral
race's 102 candidates (many earning single-digit vote totals) would make every CT look far more
fragmented than the race actually was in practice. Margin is left unscreened, matching the archived
`competition()` exactly — margin only ever looks at the top two, so a screening threshold wouldn't
change it anyway.

The municipal race needs **candidate-level** vote estimates (there's only one party bucket,
`party_non_partisan_votes`, that lumps every candidate together — useless for competitiveness), so
this section allocates the archived candidate-vote bridge through the same poll→CT crosswalk
weights already computed in Section 4, rather than re-running the spatial allocation. Provincial and
federal use the party columns already interpolated onto each CT in Section 4/6 directly.

In [9]:
def competition(values, threshold: float = 0.0):
    total = sum(values)
    if total <= 0:
        return {"margin": None, "effective": None, "fragmentation": None, "count": None}
    shares_all = sorted((v / total for v in values if v > 0), reverse=True)
    shares = [s for s in shares_all if s >= threshold]
    hhi = sum(s * s for s in shares)
    return {
        "margin": shares_all[0] - shares_all[1] if len(shares_all) > 1 else None,
        "effective": 1 / hhi if hhi else None,
        "fragmentation": 1 - hhi if hhi else None,
        "count": len(shares),
    }


# --- Municipal mayoral race: candidate-level, via the archived candidate-vote bridge ---
municipal_crosswalk = allocation_results["municipal_2023_mayor"][2]
candidate_votes = pd.read_csv(MUNICIPAL_CANDIDATE_VOTES_CSV)
candidate_lookup = pd.read_csv(MUNICIPAL_CANDIDATE_LOOKUP_CSV).set_index("candidate_id")

candidate_allocated = municipal_crosswalk.merge(candidate_votes, on="poll_id", how="inner")
candidate_allocated["estimated_candidate_votes"] = candidate_allocated["weight"] * candidate_allocated["candidate_vote_count"]
candidate_ct = (
    candidate_allocated.groupby(["ct_id", "candidate_id"])["estimated_candidate_votes"].sum().reset_index()
)
print(f"Municipal candidate-level CT allocation: {candidate_ct.shape[0]} (ct_id, candidate_id) rows, "
      f"{candidate_ct['candidate_id'].nunique()} candidates.")

block4_rows = {cid: {"ct_id": cid} for cid in ct_ids}
for cid, group in candidate_ct.groupby("ct_id"):
    values = group["estimated_candidate_votes"].tolist()
    margin_all = competition(values, 0.0)
    screened = competition(values, COMPETITIVENESS_THRESHOLD)
    row = block4_rows[cid]
    row["block4_mayoral_top_two_margin"] = margin_all["margin"]
    row["block4_mayoral_winner_margin"] = margin_all["margin"]
    row["block4_effective_mayoral_candidates_5pct"] = screened["effective"]
    row["block4_mayoral_candidate_count_5pct"] = screened["count"]
    row["block4_mayoral_vote_fragmentation"] = screened["fragmentation"]

# --- Provincial / federal: party-level, straight from the CT-level allocation ---
for election_id in ["provincial_2025", "federal_2025"]:
    short_name = ELECTIONS[election_id]["short_name"]
    ct_accum, party_fields = allocation_results[election_id][0], allocation_results[election_id][1]
    for cid, acc in ct_accum.items():
        comp = competition([acc[f] for f in party_fields], COMPETITIVENESS_THRESHOLD)
        row = block4_rows[cid]
        row[f"block4_{short_name}_margin"] = comp["margin"]
        row[f"block4_effective_{short_name}_parties_5pct"] = comp["effective"]
        row[f"block4_{short_name}_party_count_5pct"] = comp["count"]
        row[f"block4_{short_name}_vote_fragmentation"] = comp["fragmentation"]

block4_df = pd.DataFrame(block4_rows.values())
block4_df.describe()

Municipal candidate-level CT allocation: 51412 (ct_id, candidate_id) rows, 102 candidates.


,block4_mayoral_top_two_margin,block4_mayoral_winner_margin,block4_effective_mayoral_candidates_5pct,block4_mayoral_candidate_count_5pct,block4_mayoral_vote_fragmentation,block4_provincial_margin,block4_effective_provincial_parties_5pct,block4_provincial_party_count_5pct,block4_provincial_vote_fragmentation,block4_federal_margin,block4_effective_federal_parties_5pct,block4_federal_party_count_5pct,block4_federal_vote_fragmentation
count,583.000000,583.000000,583.000000,583.000000,583.000000,584.000000,584.000000,584.000000,584.000000,583.000000,583.000000,583.000000,583.000000
mean,0.164426,0.164426,3.777983,3.725557,0.725382,0.151638,2.603009,2.880137,0.609477,0.248977,2.115737,2.382504,0.525151
std,0.118856,0.118856,0.717292,0.766669,0.054159,0.122176,0.334778,0.404964,0.050051,0.131779,0.149150,0.486416,0.031452
min,0.000287,0.000287,1.625966,2.000000,0.384981,0.000024,1.543572,2.000000,0.352152,0.000436,1.819332,2.000000,0.450348
25%,0.063092,0.063092,3.217898,3.000000,0.689238,0.053856,2.355839,3.000000,0.575523,0.136854,2.031492,2.000000,0.507751
50%,0.144429,0.144429,3.779303,4.000000,0.735401,0.118205,2.517233,3.000000,0.602738,0.253049,2.091966,2.000000,0.521981
75%,0.250835,0.250835,4.252018,4.000000,0.764818,0.239958,2.893449,3.000000,0.654392,0.346787,2.162453,3.000000,0.537562
max,0.682286,0.682286,5.899683,6.000000,0.830499,0.718904,3.368323,4.000000,0.703116,0.552156,2.718806,3.000000,0.632191


## 8. Assembling and writing the final output

One row per CT (the 585-CT interpolation universe), joining turnout/participation (Section 6) with
competitiveness (Section 7) and the CT polygon (reprojected back to `EPSG:4326` to match every
other geojson in this pipeline). A separate long-format candidate/party estimated-votes table is
also written — useful if a later notebook wants finer-grained vote shares than the CT-level summary
columns carry.

In [10]:
final_df = outcome_df.merge(block4_df, on="ct_id", how="left")
final_df["ct_id"] = final_df["ct_id"].astype(float)  # match notebook 02's ct_id dtype for downstream joins

geometry_lookup = dict(zip(ct_gdf["ct_id_key"], ct_gdf.geometry))
final_gdf = gpd.GeoDataFrame(
    final_df,
    geometry=[geometry_lookup[cid] for cid in outcome_rows.keys()],
    crs=WORKING_EPSG,
).to_crs(4326)

final_df.to_csv(OUT_DIR / "ct_election_outcomes.csv", index=False)
final_gdf.to_file(OUT_DIR / "ct_election_outcomes.geojson", driver="GeoJSON")
print(f"Wrote {final_df.shape[0]} rows x {final_df.shape[1]} columns -> {OUT_DIR / 'ct_election_outcomes.csv'}")

# Long-format candidate/party estimated votes, for finer-grained downstream use.
long_rows = []
for _, r in candidate_ct.iterrows():
    candidate = candidate_lookup.loc[r["candidate_id"]]
    long_rows.append({
        "election_id": "municipal_2023_mayor", "ct_id": float(r["ct_id"]), "entity_type": "candidate",
        "entity_id": r["candidate_id"], "entity_name": candidate["candidate_name"],
        "party_name": candidate["party_name"], "estimated_votes": r["estimated_candidate_votes"],
    })
for election_id in ["provincial_2025", "federal_2025"]:
    ct_accum, party_fields = allocation_results[election_id][0], allocation_results[election_id][1]
    for cid, acc in ct_accum.items():
        for field in party_fields:
            long_rows.append({
                "election_id": election_id, "ct_id": float(cid), "entity_type": "party",
                "entity_id": field, "entity_name": field.removeprefix("party_").removesuffix("_votes"),
                "party_name": None, "estimated_votes": acc[field],
            })
long_df = pd.DataFrame(long_rows)
long_df.to_csv(OUT_DIR / "ct_candidate_party_estimated_votes.csv", index=False)
print(f"Wrote {long_df.shape[0]} rows -> {OUT_DIR / 'ct_candidate_party_estimated_votes.csv'}")

Wrote 585 rows x 27 columns -> /home/aniket/Programming/place-and-politics-toronto/data/toronto_election_turnout/interpolation/ct_election_outcomes.csv


Wrote 67792 rows -> /home/aniket/Programming/place-and-politics-toronto/data/toronto_election_turnout/interpolation/ct_candidate_party_estimated_votes.csv


## 9. Verification against the archived pipeline's ground truth

The archived GDAL/OGR-based pipeline's committed outputs are the ground truth: per-election CT
turnout/vote estimates (`interpolation/processed/*_ct_estimated_results.csv`), candidate-level
estimates (`*_ct_candidate_estimated_votes.csv`), and the Block 4/outcome columns already baked
into `variables/processed/toronto_ct_blocks_1_5_modelling_master.csv`. `ct_id` is formatted to the
same two-decimal string key (`f"{x:.2f}"`) the archived CSVs use, since floats don't compare
reliably as join keys.

Both pipelines wrap the same GEOS geometry engine (GDAL/OGR vs. shapely), so differences should be
floating-point noise from intersection ordering and `MakeValid`, not systematic — that is checked
explicitly below rather than assumed.

In [11]:
def ct_key(series):
    return series.astype(float).map(lambda x: f"{x:.2f}")


verification_passed = True

print("=" * 70)
for election_id in ELECTIONS:
    short_name = ELECTIONS[election_id]["short_name"]
    ct_accum = allocation_results[election_id][0]
    mine = pd.DataFrame([{"ct_id": cid, **acc} for cid, acc in ct_accum.items()]).set_index("ct_id")

    gt = pd.read_csv(INTERP_GT_DIR / f"{election_id}_ct_estimated_results.csv")
    gt["ct_id"] = ct_key(gt["ct_id"])
    gt = gt.set_index("ct_id")
    common = mine.index.intersection(gt.index)

    checks = {
        "estimated_total_votes": ("total_votes", 1.0),
        "estimated_electors": ("electors", 1.0),
        "estimated_valid_candidate_votes": ("valid_votes", 1.0),
    }
    print(f"{election_id}: {len(common)}/{len(gt)} CTs matched against ground truth")
    for gt_col, (my_col, atol) in checks.items():
        diff = (mine.loc[common, my_col] - gt.loc[common, gt_col]).abs()
        ok = diff.max() <= REPORT_ATOL
        verification_passed &= ok
        print(f"  {gt_col:38s} max_abs_diff={diff.max():.6e}  mean_abs_diff={diff.mean():.6e}  {'PASS' if ok else 'FAIL'}")

    my_outcome = outcome_df.set_index("ct_id")
    gt_outcome_col = f"estimated_participation_citizen_18plus"
    diff = (my_outcome.loc[common, f"outcome_{short_name}_participation_citizen_18plus"] - gt.loc[common, gt_outcome_col]).abs()
    ok = diff.max() <= 1e-6
    verification_passed &= ok
    print(f"  {'participation_citizen_18plus':38s} max_abs_diff={diff.max():.6e}  mean_abs_diff={diff.mean():.6e}  {'PASS' if ok else 'FAIL'}")
print("=" * 70)

municipal_2023_mayor: 585/585 CTs matched against ground truth
  estimated_total_votes                  max_abs_diff=1.019923e-04  mean_abs_diff=1.621927e-06  PASS
  estimated_electors                     max_abs_diff=2.280826e-04  mean_abs_diff=5.094540e-06  PASS
  estimated_valid_candidate_votes        max_abs_diff=1.019923e-04  mean_abs_diff=1.621927e-06  PASS
  participation_citizen_18plus           max_abs_diff=3.040010e-08  mean_abs_diff=5.803352e-10  PASS
provincial_2025: 585/585 CTs matched against ground truth
  estimated_total_votes                  max_abs_diff=2.127825e-04  mean_abs_diff=6.910396e-06  PASS
  estimated_electors                     max_abs_diff=2.456211e-04  mean_abs_diff=5.256695e-06  PASS
  estimated_valid_candidate_votes        max_abs_diff=2.119172e-04  mean_abs_diff=6.850675e-06  PASS
  participation_citizen_18plus           max_abs_diff=7.362716e-08  mean_abs_diff=2.251733e-09  PASS
federal_2025: 585/585 CTs matched against ground truth
  estimated_tota

In [12]:
# Municipal candidate-level votes
gt_candidates = pd.read_csv(INTERP_GT_DIR / "municipal_2023_mayor_ct_candidate_estimated_votes.csv")
gt_candidates["ct_id"] = ct_key(gt_candidates["ct_id"])
gt_idx = gt_candidates.set_index(["ct_id", "candidate_id"])["estimated_candidate_votes"]

mine_candidates = candidate_ct.copy()
mine_candidates["ct_id"] = ct_key(mine_candidates["ct_id"])
mine_idx = mine_candidates.set_index(["ct_id", "candidate_id"])["estimated_candidate_votes"]

common = gt_idx.index.intersection(mine_idx.index)
diff = (gt_idx.loc[common] - mine_idx.loc[common]).abs()
ok = diff.max() <= REPORT_ATOL and len(common) == len(gt_idx) == len(mine_idx)
verification_passed &= ok
print(f"candidate-level estimated votes: {len(common)} rows matched (gt={len(gt_idx)}, mine={len(mine_idx)})  "
      f"max_abs_diff={diff.max():.6e}  mean_abs_diff={diff.mean():.6e}  {'PASS' if ok else 'FAIL'}")

# Block 4 competitiveness, against the variables master's already-baked block4_* columns
gt_master = pd.read_csv(VARIABLES_MASTER_GT_CSV)
gt_master["ct_id"] = ct_key(gt_master["ct_id"])
gt_master = gt_master.set_index("ct_id")

my_block4 = block4_df.copy()
my_block4["ct_id"] = ct_key(my_block4["ct_id"])
my_block4 = my_block4.set_index("ct_id")

block4_cols = [c for c in my_block4.columns]
print()
for col in block4_cols:
    common = my_block4.index.intersection(gt_master.index)
    diff = (my_block4.loc[common, col] - gt_master.loc[common, col]).abs()
    ok = diff.max() <= 1e-4
    verification_passed &= ok
    print(f"  {col:45s} max_abs_diff={diff.max():.6e}  mean_abs_diff={diff.mean():.6e}  {'PASS' if ok else 'FAIL'}")

print()
print("VERIFICATION: " + ("PASS -- all interpolated turnout, vote, and competitiveness columns match the archived "
                           "pipeline's ground truth within floating-point tolerance." if verification_passed
                           else "FAIL -- see columns marked FAIL above."))
assert verification_passed, "Verification against archived ground truth failed -- see printed diffs above."

candidate-level estimated votes: 51412 rows matched (gt=51412, mine=51412)  max_abs_diff=4.303768e-05  mean_abs_diff=1.877927e-08  PASS

  block4_mayoral_top_two_margin                 max_abs_diff=7.654467e-09  mean_abs_diff=1.836790e-10  PASS
  block4_mayoral_winner_margin                  max_abs_diff=7.654467e-09  mean_abs_diff=1.836790e-10  PASS
  block4_effective_mayoral_candidates_5pct      max_abs_diff=3.001855e-08  mean_abs_diff=7.468167e-10  PASS
  block4_mayoral_candidate_count_5pct           max_abs_diff=0.000000e+00  mean_abs_diff=0.000000e+00  PASS
  block4_mayoral_vote_fragmentation             max_abs_diff=2.806781e-09  mean_abs_diff=5.964835e-11  PASS
  block4_provincial_margin                      max_abs_diff=1.946085e-08  mean_abs_diff=4.971437e-10  PASS
  block4_effective_provincial_parties_5pct      max_abs_diff=5.024511e-07  mean_abs_diff=3.650621e-09  PASS
  block4_provincial_party_count_5pct            max_abs_diff=0.000000e+00  mean_abs_diff=0.000000e+00  PASS

## Takeaways for the next notebook

- All three elections' population-weighted CT allocations reconcile with the archived GDAL/OGR
  pipeline's output to within floating-point noise (`~1e-4` votes on totals in the hundreds of
  thousands, `~1e-7` on normalized vote shares) — the shapely/GEOS reimplementation is a faithful
  port, not a re-derivation with different results.
- `outcome_mean_participation_citizen_18plus` (this notebook's headline output) and the twelve
  `block4_*` competitiveness columns are now available per CT, keyed by `ct_id`, ready to join
  against notebook 02's Block 1-3 census profile.
- The next notebook (`04_build_feature_table_and_model_input.ipynb`) joins this table's outcome +
  Block 4 columns with the Block 1-3 census profile and the still-to-be-built Block 5 (transportation/
  access/311/social-housing) variables, then median-imputes what's left missing, to produce the
  model-input table notebook 05's PLS regression will actually read.